# Corpus and Annotation Descriptives

This notebook audits the proposed manuscript table against the available project exports. It reads the current provisional split files in `prebi_v1` and, when available, the sibling `rl1_reflection_analytics` curated descriptive export and restricted MAXQDA annotation source.

The split modeling files describe an **eligible classifier view**, not the complete descriptive corpus. Do not compare their segment count directly with the historical 11,625-segment table without noting that denominator difference. This notebook displays aggregate statistics only; it never prints text, document identifiers, author names, or coder names. Raw annotation records are loaded in memory solely to calculate coder aggregates. Keep this notebook and its outputs in approved research storage.

## 1. Set Up Environment and Repository Paths

Resolve paths from the project root or its `data` directory. The sibling analytics repository is optional for split-only checks, but required to reproduce the full historical corpus and annotation table.

In [32]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "data":
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "data" / "provisional_segment_splits"
SIBLING_ROOT = PROJECT_ROOT.parent / "prebi_ap4_v1" / "rl1_reflection_analytics"
CURATED_DIR = SIBLING_ROOT / "data" / "curated"
PREPARED_DIR = CURATED_DIR / "prepared"
MODELING_DIR = CURATED_DIR / "modeling"
RAW_EXPORT = SIBLING_ROOT / "data" / "raw" / "source" / "rup_reflection_segmented_2026-09-15.xlsx"
CONTRACT_PATH = SIBLING_ROOT / "data" / "provisional_dataset_contract.json"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "corpus_annotation_descriptives"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SPLIT_DIR.is_dir(), f"Provisional split directory not found: {SPLIT_DIR}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Sibling descriptive sources available: {SIBLING_ROOT.is_dir()}")

Project root: /Users/quynguyen/Documents/CodingProjects/prebi_v1
Sibling descriptive sources available: True


## 2. Load Provisional Split Files

Read current modeling long/wide and scope files, plus split manifests and audit candidates. Columns are normalized and loaded as strings first so identifiers remain stable; numeric fields are converted only when used.

In [33]:
def read_csv_strings(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype="string", low_memory=False)
    frame.columns = frame.columns.str.strip().str.lower()
    return frame

required_split_paths = [
    SPLIT_DIR / f"provisional_{split}_{kind}.csv"
    for split in ("train", "dev", "test")
    for kind in ("modeling_wide", "modeling_long", "scope")
]
required_split_paths += [
    SPLIT_DIR / "provisional_split_manifest.csv",
    SPLIT_DIR / "provisional_split_summary.csv",
    SPLIT_DIR / "provisional_text_quality_candidates.csv",
    SPLIT_DIR / "provisional_text_quality_summary.csv",
    SPLIT_DIR / "provisional_unknown_skill_candidates.csv",
    SPLIT_DIR / "provisional_unresolved_scope_candidates.csv",
]
missing_split_files = [path for path in required_split_paths if not path.exists()]
assert not missing_split_files, "Missing required split files: " + ", ".join(map(str, missing_split_files))

split_wide = {split: read_csv_strings(SPLIT_DIR / f"provisional_{split}_modeling_wide.csv") for split in ("train", "dev", "test")}
split_long = {split: read_csv_strings(SPLIT_DIR / f"provisional_{split}_modeling_long.csv") for split in ("train", "dev", "test")}
split_scope = {split: read_csv_strings(SPLIT_DIR / f"provisional_{split}_scope.csv") for split in ("train", "dev", "test")}
split_manifest = read_csv_strings(SPLIT_DIR / "provisional_split_manifest.csv")
split_summary = read_csv_strings(SPLIT_DIR / "provisional_split_summary.csv")
quality_candidates = read_csv_strings(SPLIT_DIR / "provisional_text_quality_candidates.csv")
quality_summary = read_csv_strings(SPLIT_DIR / "provisional_text_quality_summary.csv")
unknown_skill_candidates = read_csv_strings(SPLIT_DIR / "provisional_unknown_skill_candidates.csv")
unresolved_scope_candidates = read_csv_strings(SPLIT_DIR / "provisional_unresolved_scope_candidates.csv")

split_inventory = pd.DataFrame([
    {"split": split, "candidate_rows": len(split_wide[split]), "unique_segments": split_wide[split]["segment_id"].nunique(), "long_rows": len(split_long[split]), "scope_rows": len(split_scope[split])}
    for split in ("train", "dev", "test")
])
display(split_inventory)
display(split_summary)

,split,candidate_rows,unique_segments,long_rows,scope_rows
0,train,7107,7065,21321,7107
1,dev,1823,1811,5469,1823
2,test,905,903,2715,905


,split,modeling_candidate_rows,unique_modeling_segments,documents,authors,feedback_reflections,review_required_modeling_rows,unresolved_rows_excluded,unknown_skill_rows_excluded,text_quality_rows_excluded,text_quality_segments_excluded,text_quality_only_rows_excluded,total_rows_excluded,minimum_text_characters
0,train,7107,7065,181,66,0,84,74,1368,250,190,250,1692,10
1,dev,1823,1811,34,11,0,24,14,143,76,59,76,233,10
2,test,905,903,24,8,22,4,10,188,38,30,38,236,10


## 3. Pull Supplemental Metadata from the Sibling Repository

The local split files provide document IDs, author IDs, and split metadata, but not the historical descriptive export’s creator attribution. If the sibling repository is present, join its curated prepared segment metadata to the single-label table by `segment_id` and load the raw workbook only for aggregate coder counts. Otherwise, the notebook still reports split-view counts and marks full annotation statistics unavailable.

In [34]:
sibling_sources_available = all([
    (MODELING_DIR / "provisional_single_label.csv").exists(),
    (PREPARED_DIR / "reflection_data_no_name.csv").exists(),
])

if sibling_sources_available:
    full_single = read_csv_strings(MODELING_DIR / "provisional_single_label.csv")
    prepared_segments = read_csv_strings(PREPARED_DIR / "reflection_data_no_name.csv")
    required_full_columns = {"segment_id", "text", "author_name", "semester", "seminar_name", "component", "annotation_ids"}
    assert required_full_columns.issubset(full_single.columns), f"Missing columns: {required_full_columns - set(full_single.columns)}"
    required_metadata_columns = {"segment_id", "document_id", "author_id", "semester_id", "seminar_id"}
    assert required_metadata_columns.issubset(prepared_segments.columns), f"Missing metadata columns: {required_metadata_columns - set(prepared_segments.columns)}"

    metadata_columns = ["document_id", "author_id", "semester_id", "seminar_id"]
    metadata_consistency = prepared_segments.groupby("segment_id")[metadata_columns].nunique(dropna=False)
    assert metadata_consistency.le(1).all().all(), "Prepared metadata is inconsistent within a segment"
    prepared_meta = prepared_segments[["segment_id", *metadata_columns]].drop_duplicates("segment_id")
    full_single = full_single.merge(prepared_meta, on="segment_id", how="left", validate="one_to_one")
    contract = json.loads(CONTRACT_PATH.read_text(encoding="utf-8")) if CONTRACT_PATH.exists() else {}
    legacy_contract_minimum = int(contract.get("policy", {}).get("minimum_text_length", 2))

    split_thresholds = pd.to_numeric(split_manifest["minimum_text_characters"], errors="coerce").dropna().unique()
    assert len(split_thresholds) == 1, "Expected one current minimum_text_characters value in the split manifest"
    minimum_text_length = int(split_thresholds[0])

    annotation_ready = RAW_EXPORT.exists()
    if annotation_ready:
        from importlib.util import module_from_spec, spec_from_file_location
        from typing import Callable, cast

        utility_path = SIBLING_ROOT / "utils" / "canonical_annotations.py"
        utility_spec = spec_from_file_location("canonical_annotations", utility_path)
        if utility_spec is None or utility_spec.loader is None:
            raise ImportError(f"Could not load annotation utility: {utility_path}")
        utility_module = module_from_spec(utility_spec)
        utility_spec.loader.exec_module(utility_module)
        build_annotations = cast(Callable[[pd.DataFrame], pd.DataFrame], getattr(utility_module, "build_canonical_annotations"))

        raw_annotations = pd.read_excel(RAW_EXPORT)
        canonical_annotations = build_annotations(raw_annotations)
        coder_values = set(canonical_annotations["creator"].dropna()) | set(canonical_annotations["editor"].dropna())
        annotator_count = len(coder_values)
        creator_by_annotation = canonical_annotations.set_index("annotation_id")["creator"]
        del raw_annotations
    else:
        annotator_count = np.nan
        creator_by_annotation = pd.Series(dtype="string")
else:
    full_single = pd.DataFrame()
    prepared_segments = pd.DataFrame()
    contract = {}
    legacy_contract_minimum = np.nan
    minimum_text_length = np.nan
    annotation_ready = False
    annotator_count = np.nan
    creator_by_annotation = pd.Series(dtype="string")

print(f"Sibling curated sources available: {sibling_sources_available}")
print(f"Restricted annotation workbook available: {annotation_ready}")
if sibling_sources_available:
    print(f"Full single-label rows: {len(full_single):,}")
    print(f"Current provisional split threshold: {minimum_text_length} characters")
    print(f"Legacy sibling contract threshold: {legacy_contract_minimum} characters")

Sibling curated sources available: True
Restricted annotation workbook available: True
Full single-label rows: 11,625
Current provisional split threshold: 10 characters
Legacy sibling contract threshold: 2 characters


## 4. Assemble Canonical Segment-Level Analysis Tables

Retain two clearly labeled views: the local split modeling view (candidate rows deduplicated to segments) and the sibling full descriptive single-label view. The prepared metadata join supplies document and stable author identifiers; annotation references and scope are retained from the corresponding source. Segment IDs are used only in memory and are never displayed.

In [35]:
local_candidates = pd.concat(
    [frame.assign(split=split) for split, frame in split_wide.items()],
    ignore_index=True,
)
local_group_checks = local_candidates.groupby("segment_id")[
    [column for column in ("document_id", "author_id", "split") if column in local_candidates.columns]
].nunique(dropna=False)
assert local_group_checks.le(1).all().all(), "A segment crosses document, author, or split boundaries"
local_segments = local_candidates.drop_duplicates("segment_id").copy()

if sibling_sources_available:
    analysis_segments = full_single.drop_duplicates("segment_id").copy()
    assert analysis_segments["segment_id"].is_unique, "Full descriptive view must have one row per segment"
    metadata_coverage = analysis_segments["document_id"].notna().mean()
    assert metadata_coverage == 1.0, "Some descriptive segments lack document metadata"
else:
    analysis_segments = local_segments.copy()
    metadata_coverage = np.nan

view_inventory = pd.DataFrame([
    {"view": "Current provisional split modeling view", "candidate_rows": len(local_candidates), "unique_segments": local_segments["segment_id"].nunique(), "documents": local_candidates["document_id"].nunique(), "authors": local_candidates["author_id"].nunique()},
    {"view": "Sibling full descriptive view", "candidate_rows": len(full_single) if sibling_sources_available else np.nan, "unique_segments": analysis_segments["segment_id"].nunique(), "documents": analysis_segments["document_id"].nunique() if sibling_sources_available else np.nan, "authors": analysis_segments["author_name"].nunique() if sibling_sources_available else np.nan},
])
display(view_inventory)
print(f"Prepared metadata coverage of full descriptive segments: {metadata_coverage:.1%}" if sibling_sources_available else "Prepared metadata not available; reporting split view only")

,view,candidate_rows,unique_segments,documents,authors
0,Current provisional split modeling view,9835,9779,239,85
1,Sibling full descriptive view,11625,11625,239,63


Prepared metadata coverage of full descriptive segments: 100.0%


## 5. Compute Corpus Characteristics

Calculate counts from unique segment-level records and document/author metadata. Counts from the present provisional split manifest are also shown because its author IDs and eligibility rules differ from the sibling historical descriptive export.

In [36]:
def unique_count(frame: pd.DataFrame, column: str) -> int:
    return int(frame[column].nunique(dropna=True)) if column in frame.columns else 0

corpus_stats = {
    "documents": unique_count(analysis_segments, "document_id"),
    "authors": unique_count(analysis_segments, "author_name" if sibling_sources_available else "author_id"),
    "segments": unique_count(analysis_segments, "segment_id"),
    "seminars": unique_count(analysis_segments, "seminar_name" if sibling_sources_available else "seminar_id"),
    "semesters": unique_count(analysis_segments, "semester" if sibling_sources_available else "semester_id"),
}
local_corpus_stats = {
    "documents": unique_count(local_candidates, "document_id"),
    "authors": unique_count(local_candidates, "author_id"),
    "segments": unique_count(local_segments, "segment_id"),
}
corpus_comparison = pd.DataFrame([
    {"statistic": "documents", "full descriptive export": corpus_stats["documents"], "current split view": local_corpus_stats["documents"]},
    {"statistic": "authors", "full descriptive export": corpus_stats["authors"], "current split view": local_corpus_stats["authors"]},
    {"statistic": "segments", "full descriptive export": corpus_stats["segments"], "current split view": local_corpus_stats["segments"]},
    {"statistic": "seminars / semesters", "full descriptive export": f"{corpus_stats['seminars']} / {corpus_stats['semesters']}", "current split view": f"{unique_count(local_candidates, 'seminar_id')} / {unique_count(local_candidates, 'semester_id')}"},
])
display(corpus_comparison)
print("Full descriptive view is backed by sibling curated files." if sibling_sources_available else "Full descriptive view unavailable; values above fall back to provisional splits.")

,statistic,full descriptive export,current split view
0,documents,239,239
1,authors,63,85
2,segments,11625,9779
3,seminars / semesters,2 / 5,2 / 5


Full descriptive view is backed by sibling curated files.


## 6. Compute Granularity Statistics

Use one row per segment for both distributions. Document-level segment counts are grouped using `document_id`; token counts use whitespace-separated tokens, matching the convention in the sibling descriptive report. The current split manifest sets a 10-character minimum, while the older sibling contract records 2; both filtered views are reported because they describe different artifact generations.

In [37]:
def median_iqr(values: pd.Series) -> tuple[float, float, float]:
    numeric = pd.to_numeric(values, errors="coerce").dropna()
    return (float(numeric.median()), float(numeric.quantile(0.25)), float(numeric.quantile(0.75)))

def format_median_iqr(values: pd.Series) -> str:
    median, q1, q3 = median_iqr(values)
    return f"{median:.0f} [{q1:.0f}--{q3:.0f}]"

def whitespace_token_count(text: object) -> int:
    return len(str(text).split())

analysis_segments["word_count"] = analysis_segments["text"].fillna("").map(whitespace_token_count)
document_segment_counts = analysis_segments.groupby("document_id", dropna=True)["segment_id"].nunique()
segment_length_stats = {
    "segments_per_document": format_median_iqr(document_segment_counts),
    "words_per_segment": format_median_iqr(analysis_segments["word_count"]),
}

if sibling_sources_available:
    text_lengths = full_single["text"].fillna("").str.len()
    policy_segments = full_single.loc[text_lengths.ge(minimum_text_length)].copy()
    policy_segments["word_count"] = policy_segments["text"].fillna("").map(whitespace_token_count)
    policy_doc_counts = policy_segments.groupby("document_id", dropna=True)["segment_id"].nunique()
    policy_granularity = {
        "segments": policy_segments["segment_id"].nunique(),
        "segments_per_document": format_median_iqr(policy_doc_counts),
        "words_per_segment": format_median_iqr(policy_segments["word_count"]),
        "excluded_short_segments": int((~text_lengths.ge(minimum_text_length)).sum()),
    }

    legacy_policy_segments = full_single.loc[text_lengths.ge(legacy_contract_minimum)].copy()
    legacy_policy_segments["word_count"] = legacy_policy_segments["text"].fillna("").map(whitespace_token_count)
    legacy_doc_counts = legacy_policy_segments.groupby("document_id", dropna=True)["segment_id"].nunique()
    legacy_contract_granularity = {
        "segments": legacy_policy_segments["segment_id"].nunique(),
        "segments_per_document": format_median_iqr(legacy_doc_counts),
        "words_per_segment": format_median_iqr(legacy_policy_segments["word_count"]),
        "excluded_short_segments": int((~text_lengths.ge(legacy_contract_minimum)).sum()),
    }
else:
    policy_segments = analysis_segments.copy()
    legacy_policy_segments = analysis_segments.copy()
    policy_granularity = {
        "segments": len(analysis_segments),
        "segments_per_document": segment_length_stats["segments_per_document"],
        "words_per_segment": segment_length_stats["words_per_segment"],
        "excluded_short_segments": np.nan,
    }
    legacy_contract_granularity = policy_granularity.copy()

granularity_comparison = pd.DataFrame([
    {"statistic": "segments per document, median [IQR]", "stored export": segment_length_stats["segments_per_document"], "current split policy": policy_granularity["segments_per_document"], "legacy contract policy": legacy_contract_granularity["segments_per_document"]},
    {"statistic": "words per segment, median [IQR]", "stored export": segment_length_stats["words_per_segment"], "current split policy": policy_granularity["words_per_segment"], "legacy contract policy": legacy_contract_granularity["words_per_segment"]},
    {"statistic": "segments retained", "stored export": len(analysis_segments), "current split policy": policy_granularity["segments"], "legacy contract policy": legacy_contract_granularity["segments"]},
    {"statistic": f"segments shorter than {minimum_text_length} characters", "stored export": policy_granularity["excluded_short_segments"], "current split policy": 0, "legacy contract policy": np.nan},
    {"statistic": f"segments shorter than {legacy_contract_minimum} characters", "stored export": legacy_contract_granularity["excluded_short_segments"], "current split policy": np.nan, "legacy contract policy": 0},
])
display(granularity_comparison)

,statistic,stored export,current split policy,legacy contract policy
0,"segments per document, median [IQR]",47 [31--64],46 [30--63],46 [30--63]
1,"words per segment, median [IQR]",18 [12--24],18 [13--24],18 [13--24]
2,segments retained,11625,11420,11433
3,segments shorter than 10 characters,205,0,NaN
4,segments shorter than 2 characters,192,NaN,0


## 7. Compute Annotation Workforce and Multi-Annotator Coverage

The workforce count is the union of nonmissing creators and editors, matching the sibling report’s definition. Multi-annotator coverage counts distinct **creators** referenced by each segment’s `annotation_ids`; editor-only revision history is not treated as a second annotation.

In [45]:
if sibling_sources_available and annotation_ready:
    creator_sets = {}
    single_creator_by_segment = {}
    for row in full_single[["segment_id", "annotation_ids"]].itertuples(index=False):
        annotation_ids = [] if pd.isna(row.annotation_ids) else str(row.annotation_ids).split("|")
        creators = creator_by_annotation.reindex(annotation_ids).dropna().unique()
        creator_sets[row.segment_id] = len(creators)
        if len(creators) == 1:
            single_creator_by_segment[row.segment_id] = creators[0]

    creator_counts = pd.Series(creator_sets, name="distinct_creators")
    multi_annotator_segments = int(creator_counts.gt(1).sum())
    single_annotator_segments = int(creator_counts.eq(1).sum())
    annotated_segment_count = int(creator_counts.size)
    multi_annotator_pct = 100 * multi_annotator_segments / annotated_segment_count
    single_annotator_pct = 100 * single_annotator_segments / annotated_segment_count
    coder_aliases = {name: f"C{index:02d}" for index, name in enumerate(sorted(coder_values), start=1)}
    single_coder_segments = pd.Series(single_creator_by_segment, name="creator").map(coder_aliases)

    creator_count_distribution = (
        creator_counts.value_counts().sort_index().rename_axis("distinct_creator_count")
        .reset_index(name="segments")
    )
    creator_count_distribution["share_pct"] = (
        100 * creator_count_distribution["segments"] / annotated_segment_count
    )
    single_coder_distribution = (
        single_coder_segments.value_counts().sort_index().rename_axis("coder")
        .reset_index(name="segments_coded_alone")
    )
    single_coder_distribution["share_of_single_coded_pct"] = (
        100 * single_coder_distribution["segments_coded_alone"] / single_annotator_segments
    )
    single_coder_distribution["share_of_all_segments_pct"] = (
        100 * single_coder_distribution["segments_coded_alone"] / annotated_segment_count
    )
else:
    creator_counts = pd.Series(dtype="int64", name="distinct_creators")
    single_coder_segments = pd.Series(dtype="string", name="creator")
    creator_count_distribution = pd.DataFrame(columns=["distinct_creator_count", "segments", "share_pct"])
    single_coder_distribution = pd.DataFrame(columns=["coder", "segments_coded_alone", "share_of_single_coded_pct", "share_of_all_segments_pct"])
    multi_annotator_segments = np.nan
    single_annotator_segments = np.nan
    annotated_segment_count = np.nan
    multi_annotator_pct = np.nan
    single_annotator_pct = np.nan

annotation_stats = pd.DataFrame([
    {"statistic": "distinct annotator identities (creator/editor union)", "value": annotator_count},
    {"statistic": "segments coded by exactly 1 distinct creator", "value": single_annotator_segments},
    {"statistic": "segments coded by exactly 1 distinct creator (%)", "value": round(single_annotator_pct, 1) if pd.notna(single_annotator_pct) else np.nan},
    {"statistic": "segments coded by >1 distinct creator", "value": multi_annotator_segments},
    {"statistic": "segments coded by >1 distinct creator (%)", "value": round(multi_annotator_pct, 1) if pd.notna(multi_annotator_pct) else np.nan},
    {"statistic": "segment denominator", "value": annotated_segment_count},
])
display(annotation_stats)
if pd.notna(multi_annotator_pct):
    print(f"Exactly one creator = {single_annotator_segments:,} / {annotated_segment_count:,} = {single_annotator_pct:.1f}%")
    print(f"More than one creator = {multi_annotator_segments:,} / {annotated_segment_count:,} = {multi_annotator_pct:.1f}%")

,statistic,value
0,distinct annotator identities (creator/editor ...,6.0
1,segments coded by exactly 1 distinct creator,10694.0
2,segments coded by exactly 1 distinct creator (%),92.0
3,segments coded by >1 distinct creator,931.0
4,segments coded by >1 distinct creator (%),8.0
5,segment denominator,11625.0


Exactly one creator = 10,694 / 11,625 = 92.0%
More than one creator = 931 / 11,625 = 8.0%


### Distribution by Number of Annotators

First show how many distinct creators contributed to each segment. Then break down the segments with exactly one creator by anonymized coder label. The single-coder percentages are shown both within the single-coded subset and against all segments; raw coder identities are not displayed.

In [49]:
if sibling_sources_available and annotation_ready:
    creator_count_distribution["annotator_group"] = creator_count_distribution["distinct_creator_count"].map(
        lambda count: "1 annotator (single-coded)" if count == 1 else f"{count} annotators"
    )
    creator_count_distribution["share_pct"] = (
        100 * creator_count_distribution["segments"] / annotated_segment_count
    ).round(2)
    creator_count_display = creator_count_distribution[
        ["annotator_group", "segments", "share_pct"]
    ]

    all_coder_labels = sorted(coder_aliases.values())
    single_coder_distribution = (
        single_coder_segments.value_counts().reindex(all_coder_labels, fill_value=0)
        .rename_axis("coder").reset_index(name="segments_coded_alone")
    )
    single_coder_distribution["share_of_single_coded_pct"] = (
        100 * single_coder_distribution["segments_coded_alone"] / single_annotator_segments
    ).round(1)
    single_coder_distribution["share_of_all_segments_pct"] = (
        100 * single_coder_distribution["segments_coded_alone"] / annotated_segment_count
    ).round(1)
    single_coder_display = single_coder_distribution.sort_values(
        "coder", kind="stable"
    ).reset_index(drop=True)

    single_coded_total = int(creator_counts.eq(1).sum())
    assert int(creator_count_distribution["segments"].sum()) == annotated_segment_count
    assert int(single_coder_distribution["segments_coded_alone"].sum()) == single_coded_total
    assert len(single_coder_distribution) == annotator_count
    assert np.isclose(single_coder_distribution["share_of_single_coded_pct"].sum(), 100.0, atol=0.1)
    assert single_coded_total == 10694

    print(f"Segment distribution (denominator N={annotated_segment_count:,}):")
    display(creator_count_display)
    print(f"Single-coded segment distribution by anonymized creator (n={single_coded_total:,}):")
    display(single_coder_display)
else:
    print("Creator-count and single-coder distributions are unavailable without the sibling annotations.")

Segment distribution (denominator N=11,625):


,annotator_group,segments,share_pct
0,1 annotator (single-coded),10694,91.99
1,2 annotators,927,7.97
2,3 annotators,4,0.03


Single-coded segment distribution by anonymized creator (n=10,694):


,coder,segments_coded_alone,share_of_single_coded_pct,share_of_all_segments_pct
0,C01,76,0.7,0.7
1,C02,0,0.0,0.0
2,C03,1068,10.0,9.2
3,C04,472,4.4,4.1
4,C05,4456,41.7,38.3
5,C06,4622,43.2,39.8


## 8. Compute Scope Distribution

Use the segment-level `component` label in the historical single-label view. The terms “reflection-related” and “non-reflection-related” are manuscript labels for `in_scope` and `out_of_scope`, respectively; they are not inferred from text.

In [39]:
def scope_summary(frame: pd.DataFrame) -> dict[str, int | float]:
    labels = frame["component"].fillna("").astype(str).str.strip().str.lower()
    reflection_mask = labels.isin({"in_scope", "in-scope", "reflection_related", "reflection-related"})
    nonreflection_mask = labels.isin({"out_of_scope", "out-of-scope", "non_reflection_related", "non-reflection-related"})
    total = int(frame["segment_id"].nunique())
    reflection_n = int(reflection_mask.sum())
    nonreflection_n = int(nonreflection_mask.sum())
    unclassified_n = total - reflection_n - nonreflection_n
    return {
        "total": total,
        "reflection": reflection_n,
        "nonreflection": nonreflection_n,
        "unclassified": unclassified_n,
        "reflection_pct": 100 * reflection_n / total if total else np.nan,
        "nonreflection_pct": 100 * nonreflection_n / total if total else np.nan,
    }

if sibling_sources_available:
    scope_full = scope_summary(full_single)
    scope_policy = scope_summary(policy_segments)
    scope_legacy_policy = scope_summary(legacy_policy_segments)
else:
    scope_full = {key: np.nan for key in ("total", "reflection", "nonreflection", "unclassified", "reflection_pct", "nonreflection_pct")}
    scope_policy = scope_full.copy()
    scope_legacy_policy = scope_full.copy()

scope_comparison = pd.DataFrame([
    {"scope": "reflection-related", "stored export": scope_full["reflection"], "stored export (%)": round(scope_full["reflection_pct"], 1) if pd.notna(scope_full["reflection_pct"]) else np.nan, "current split policy": scope_policy["reflection"], "current policy (%)": round(scope_policy["reflection_pct"], 1) if pd.notna(scope_policy["reflection_pct"]) else np.nan, "legacy contract policy": scope_legacy_policy["reflection"], "legacy policy (%)": round(scope_legacy_policy["reflection_pct"], 1) if pd.notna(scope_legacy_policy["reflection_pct"]) else np.nan},
    {"scope": "non-reflection-related", "stored export": scope_full["nonreflection"], "stored export (%)": round(scope_full["nonreflection_pct"], 1) if pd.notna(scope_full["nonreflection_pct"]) else np.nan, "current split policy": scope_policy["nonreflection"], "current policy (%)": round(scope_policy["nonreflection_pct"], 1) if pd.notna(scope_policy["nonreflection_pct"]) else np.nan, "legacy contract policy": scope_legacy_policy["nonreflection"], "legacy policy (%)": round(scope_legacy_policy["nonreflection_pct"], 1) if pd.notna(scope_legacy_policy["nonreflection_pct"]) else np.nan},
    {"scope": "unclassified", "stored export": scope_full["unclassified"], "stored export (%)": np.nan, "current split policy": scope_policy["unclassified"], "current policy (%)": np.nan, "legacy contract policy": scope_legacy_policy["unclassified"], "legacy policy (%)": np.nan},
])
display(scope_comparison)

,scope,stored export,stored export (%),current split policy,current policy (%),legacy contract policy,legacy policy (%)
0,reflection-related,11164,96.0,10965,96.0,10972,96.0
1,non-reflection-related,461,4.0,455,4.0,461,4.0
2,unclassified,0,NaN,0,NaN,0,NaN


## 9. Recreate the Descriptive Statistics Table

Generate the requested table from the computed values. The displayed count/percentage table uses the stored export to make it comparable to the proposed manuscript values; policy-filtered alternatives remain visible in the preceding sections.

In [40]:
def comma(value: int | float) -> str:
    return f"{int(value):,}" if pd.notna(value) else "Unavailable"

def count_percent(count: int | float, pct: float) -> str:
    return f"{comma(count)} ({pct:.1f}%)" if pd.notna(count) and pd.notna(pct) else "Unavailable"

seminar_semester_value = f"{corpus_stats['seminars']} / {corpus_stats['semesters']}" if sibling_sources_available else "Unavailable"
annotator_value = comma(annotator_count)
multi_annotator_value = count_percent(multi_annotator_segments, multi_annotator_pct) if pd.notna(multi_annotator_pct) else "Unavailable"
reflection_value = count_percent(scope_full["reflection"], scope_full["reflection_pct"]) if pd.notna(scope_full["reflection_pct"]) else "Unavailable"
nonreflection_value = count_percent(scope_full["nonreflection"], scope_full["nonreflection_pct"]) if pd.notna(scope_full["nonreflection_pct"]) else "Unavailable"

descriptive_table = pd.DataFrame([
    {"Aspect": "Corpus", "Statistic": "Documents", "Value": comma(corpus_stats["documents"])},
    {"Aspect": "", "Statistic": "Authors", "Value": comma(corpus_stats["authors"])},
    {"Aspect": "", "Statistic": "Segments", "Value": comma(corpus_stats["segments"])},
    {"Aspect": "", "Statistic": "Seminars / semesters", "Value": seminar_semester_value},
    {"Aspect": "Granularity", "Statistic": "Segments per document, median [IQR]", "Value": segment_length_stats["segments_per_document"]},
    {"Aspect": "", "Statistic": "Word count per segment, median [IQR]", "Value": segment_length_stats["words_per_segment"]},
    {"Aspect": "Annotation", "Statistic": "Annotators", "Value": annotator_value},
    {"Aspect": "", "Statistic": "Segments coded by >1 annotator", "Value": multi_annotator_value},
    {"Aspect": "Scope", "Statistic": "Reflection-related segments", "Value": reflection_value},
    {"Aspect": "", "Statistic": "Non-reflection-related segments", "Value": nonreflection_value},
])
display(descriptive_table)

latex_table = descriptive_table.to_latex(index=False, escape=True, column_format="lll")
latex_table = latex_table.replace(
    r"\begin{tabular}{lll}",
    r"\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}p{0.18\linewidth}>{\raggedright\arraybackslash}X>{\raggedleft\arraybackslash}p{0.18\linewidth}@{}}",
).replace(r"\end{tabular}", r"\end{tabularx}")
latex_table = latex_table.replace(r"\toprule", r"\hline").replace(r"\midrule", r"\hline").replace(r"\bottomrule", r"\hline")
print("LaTeX-ready table (requires tabularx and array in the document preamble):")
print(r"\begin{table}[!htbp]")
print(r"\centering")
print(r"\caption{Corpus and annotation characteristics.}")
print(r"\label{tab:data}")
print(latex_table)
print(r"\end{table}")

,Aspect,Statistic,Value
0,Corpus,Documents,239
1,,Authors,63
2,,Segments,"11,625"
3,,Seminars / semesters,2 / 5
4,Granularity,"Segments per document, median [IQR]",47 [31--64]
5,,"Word count per segment, median [IQR]",18 [12--24]
6,Annotation,Annotators,6
7,,Segments coded by >1 annotator,931 (8.0%)
8,Scope,Reflection-related segments,"11,164 (96.0%)"
9,,Non-reflection-related segments,461 (4.0%)


LaTeX-ready table (requires tabularx and array in the document preamble):
\begin{table}[!htbp]
\centering
\caption{Corpus and annotation characteristics.}
\label{tab:data}
\begin{tabularx}{\linewidth}{@{}>{\raggedright\arraybackslash}p{0.18\linewidth}>{\raggedright\arraybackslash}X>{\raggedleft\arraybackslash}p{0.18\linewidth}@{}}
\hline
Aspect & Statistic & Value \\
\hline
Corpus & Documents & 239 \\
 & Authors & 63 \\
 & Segments & 11,625 \\
 & Seminars / semesters & 2 / 5 \\
Granularity & Segments per document, median [IQR] & 47 [31--64] \\
 & Word count per segment, median [IQR] & 18 [12--24] \\
Annotation & Annotators & 6 \\
 & Segments coded by >1 annotator & 931 (8.0\%) \\
Scope & Reflection-related segments & 11,164 (96.0\%) \\
 & Non-reflection-related segments & 461 (4.0\%) \\
\hline
\end{tabularx}

\end{table}


## 10. Add Automated Consistency Checks and Export Outputs

Checks compare the calculated stored-export statistics with the supplied values, while invariant assertions protect denominators and known artifact counts. A mismatch is reported rather than overwritten. Exports contain aggregates only.

In [41]:
proposed = {
    "documents": 239,
    "authors": 63,
    "segments": 11625,
    "seminars / semesters": "2 / 8",
    "segments per document, median [IQR]": "47 [31--64]",
    "word count per segment, median [IQR]": "18 [12--24]",
    "annotators": 6,
    "segments coded by >1 annotator": "931 (8.0%)",
    "reflection-related segments": "11,164 (96.0%)",
    "non-reflection-related segments": "461 (4.0%)",
}
calculated = {
    "documents": comma(corpus_stats["documents"]),
    "authors": comma(corpus_stats["authors"]),
    "segments": comma(corpus_stats["segments"]),
    "seminars / semesters": seminar_semester_value,
    "segments per document, median [IQR]": segment_length_stats["segments_per_document"],
    "word count per segment, median [IQR]": segment_length_stats["words_per_segment"],
    "annotators": annotator_value,
    "segments coded by >1 annotator": multi_annotator_value,
    "reflection-related segments": reflection_value,
    "non-reflection-related segments": nonreflection_value,
}
match_flags = {
    "documents": corpus_stats["documents"] == proposed["documents"],
    "authors": corpus_stats["authors"] == proposed["authors"],
    "segments": corpus_stats["segments"] == proposed["segments"],
    "seminars / semesters": seminar_semester_value == proposed["seminars / semesters"],
    "segments per document, median [IQR]": segment_length_stats["segments_per_document"] == proposed["segments per document, median [IQR]"],
    "word count per segment, median [IQR]": segment_length_stats["words_per_segment"] == proposed["word count per segment, median [IQR]"],
    "annotators": annotator_count == proposed["annotators"],
    "segments coded by >1 annotator": multi_annotator_segments == 931 and np.isclose(multi_annotator_pct, 8.0, atol=0.05),
    "reflection-related segments": scope_full["reflection"] == 11164 and np.isclose(scope_full["reflection_pct"], 96.0, atol=0.05),
    "non-reflection-related segments": scope_full["nonreflection"] == 461 and np.isclose(scope_full["nonreflection_pct"], 4.0, atol=0.05),
}
consistency_checks = pd.DataFrame([
    {"statistic": key, "proposed": proposed[key], "calculated": calculated[key], "matches": bool(match_flags[key])}
    for key in proposed
])
display(consistency_checks)

if sibling_sources_available and annotation_ready:
    assert corpus_stats["documents"] == 239
    assert corpus_stats["authors"] == 63
    assert corpus_stats["segments"] == 11625
    assert corpus_stats["seminars"] == 2
    assert corpus_stats["semesters"] == 5
    assert segment_length_stats["segments_per_document"] == "47 [31--64]"
    assert segment_length_stats["words_per_segment"] == "18 [12--24]"
    assert annotator_count == 6
    assert multi_annotator_segments == 931 and np.isclose(multi_annotator_pct, 8.0, atol=0.05)
    assert scope_full["reflection"] == 11164 and scope_full["nonreflection"] == 461
    assert np.isclose(scope_full["reflection_pct"], 96.0, atol=0.05)
    assert np.isclose(scope_full["nonreflection_pct"], 4.0, atol=0.05)
    assert minimum_text_length == 10
    assert legacy_contract_minimum == 2
    assert policy_granularity["segments"] == len(full_single) - policy_granularity["excluded_short_segments"]
    assert legacy_contract_granularity["segments"] == 11433
    assert legacy_contract_granularity["excluded_short_segments"] == 192
    assert not all(consistency_checks["matches"])
    assert consistency_checks.loc[consistency_checks["statistic"].eq("seminars / semesters"), "matches"].item() == False
    print("Verified corpus, annotation, scope, and policy baselines.")
    print("Only submitted-table mismatch: curated metadata reports 2 seminars / 5 semesters, not 2 / 8.")
    print(f"Current split policy uses a {minimum_text_length}-character minimum; the older sibling contract records {legacy_contract_minimum} characters.")
else:
    assert local_corpus_stats["segments"] == sum(int(value) for value in split_summary["unique_modeling_segments"])
    print("Split-view invariant checks passed; sibling-only annotation statistics were unavailable.")

,statistic,proposed,calculated,matches
0,documents,239,239,True
1,authors,63,63,True
2,segments,11625,"11,625",True
3,seminars / semesters,2 / 8,2 / 5,False
4,"segments per document, median [IQR]",47 [31--64],47 [31--64],True
5,"word count per segment, median [IQR]",18 [12--24],18 [12--24],True
6,annotators,6,6,True
7,segments coded by >1 annotator,931 (8.0%),931 (8.0%),True
8,reflection-related segments,"11,164 (96.0%)","11,164 (96.0%)",True
9,non-reflection-related segments,461 (4.0%),461 (4.0%),True


Verified corpus, annotation, scope, and policy baselines.
Only submitted-table mismatch: curated metadata reports 2 seminars / 5 semesters, not 2 / 8.
Current split policy uses a 10-character minimum; the older sibling contract records 2 characters.


In [50]:
summary = {
    "source_views": {
        "current_split_unique_segments": int(local_corpus_stats["segments"]),
        "current_split_documents": int(local_corpus_stats["documents"]),
        "current_split_author_ids": int(local_corpus_stats["authors"]),
        "historical_full_segments": int(corpus_stats["segments"]),
        "historical_documents": int(corpus_stats["documents"]),
        "historical_authors": int(corpus_stats["authors"]),
    },
    "historical_corpus": {
        "seminars": int(corpus_stats["seminars"]),
        "semesters": int(corpus_stats["semesters"]),
        "segments_per_document_median_iqr": segment_length_stats["segments_per_document"],
        "words_per_segment_median_iqr": segment_length_stats["words_per_segment"],
        "annotators_creator_editor_union": int(annotator_count),
        "single_creator_segments": int(single_annotator_segments),
        "single_creator_percent": round(float(single_annotator_pct), 3),
        "multi_creator_segments": int(multi_annotator_segments),
        "multi_creator_percent": round(float(multi_annotator_pct), 3),
        "creator_count_distribution": creator_count_distribution[
            ["distinct_creator_count", "segments", "share_pct"]
        ].to_dict(orient="records"),
        "single_creator_distribution_by_pseudonym": single_coder_distribution.to_dict(orient="records"),
        "reflection_related_segments": int(scope_full["reflection"]),
        "non_reflection_related_segments": int(scope_full["nonreflection"]),
    },
    "current_split_policy": {
        "minimum_text_characters": int(minimum_text_length),
        "retained_segments_in_historical_export": int(policy_granularity["segments"]),
        "excluded_short_segments": int(policy_granularity["excluded_short_segments"]),
        "reflection_related": int(scope_policy["reflection"]),
        "non_reflection_related": int(scope_policy["nonreflection"]),
    },
    "legacy_sibling_contract": {
        "minimum_text_characters": int(legacy_contract_minimum),
        "retained_segments": int(legacy_contract_granularity["segments"]),
        "excluded_short_segments": int(legacy_contract_granularity["excluded_short_segments"]),
        "reflection_related": int(scope_legacy_policy["reflection"]),
        "non_reflection_related": int(scope_legacy_policy["nonreflection"]),
    },
    "proposed_table_matches": {str(row.statistic): bool(row.matches) for row in consistency_checks.itertuples(index=False)},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
descriptive_table.to_csv(OUTPUT_DIR / "corpus_annotation_descriptive_table.csv", index=False)
consistency_checks.to_csv(OUTPUT_DIR / "corpus_annotation_consistency_checks.csv", index=False)
creator_count_distribution.to_csv(OUTPUT_DIR / "segment_annotator_count_distribution.csv", index=False)
single_coder_distribution.to_csv(OUTPUT_DIR / "single_coder_segment_distribution_pseudonymized.csv", index=False)
(OUTPUT_DIR / "corpus_annotation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
latex_fragment = "\n".join([
    r"\begin{table}[!htbp]",
    r"\centering",
    r"\caption{Corpus and annotation characteristics.}",
    r"\label{tab:data}",
    latex_table,
    r"\end{table}",
    "",
])
(OUTPUT_DIR / "corpus_annotation_descriptive_table.tex").write_text(
    "% Requires tabularx and array in the document preamble.\n" + latex_fragment,
    encoding="utf-8",
)
print(f"Aggregate-only outputs saved under: {OUTPUT_DIR}")
print("Exports include overall annotator-count and pseudonymized single-creator distributions; raw coder names are not written.")

Aggregate-only outputs saved under: /Users/quynguyen/Documents/CodingProjects/prebi_v1/artifacts/corpus_annotation_descriptives
Exports include overall annotator-count and pseudonymized single-creator distributions; raw coder names are not written.


## 11. Experimental Train/Dev/Test Split

This section describes the **current experimental modeling split**, not the historical descriptive corpus. Counts are calculated from the canonical `modeling_wide` exports. Candidate rows are the modeling unit; unique segments are also reported because a segment can have multiple candidate annotation bundles. Documents were kept intact across partitions. Author counts use normalized author names rather than source `author_id` values; the split was originally assigned using source IDs, so normalized-name overlap across partitions is reported explicitly. Per-split author counts therefore overlap and need not sum to the 63 unique authors overall. The 22 feedback-available reflections are assigned to test. Text-quality exclusions use the split manifest's 10-character threshold; unresolved-scope and unknown-skill candidates are excluded from the common modeling view.

In [52]:
if sibling_sources_available:
    author_name_map = local_segments[["segment_id", "author_id"]].merge(
        full_single[["segment_id", "author_name"]],
        on="segment_id",
        how="inner",
        validate="one_to_one",
    )
    author_name_map["normalized_author"] = (
        author_name_map["author_name"].astype("string").str.strip().str.casefold()
        .str.replace(r"\s+", " ", regex=True)
    )
    author_mapping_counts = author_name_map.groupby("author_id")["normalized_author"].nunique()
    assert author_mapping_counts.eq(1).all(), "A source author ID maps to multiple normalized names"
    normalized_author_lookup = author_name_map.drop_duplicates("author_id").set_index("author_id")["normalized_author"]
    local_segments["normalized_author"] = local_segments["author_id"].map(normalized_author_lookup)
    assert local_segments["normalized_author"].notna().all(), "Some current split author IDs lack a normalized-name mapping"
    normalized_authors_cross_split = int(
        local_segments.groupby("normalized_author")["split"].nunique().gt(1).sum()
    )
else:
    local_segments["normalized_author"] = pd.NA
    normalized_authors_cross_split = np.nan

experimental_split_rows = []
for split_name in ("train", "dev", "test"):
    candidate_frame = split_wide[split_name]
    split_segments = local_segments.loc[local_segments["split"].eq(split_name)]
    split_summary_row = split_summary.loc[split_summary["split"].eq(split_name)].iloc[0]
    segment_count = int(candidate_frame["segment_id"].nunique())
    candidate_count = len(candidate_frame)
    document_count = int(pd.to_numeric(split_summary_row["documents"]))
    normalized_author_count = int(split_segments["normalized_author"].nunique()) if sibling_sources_available else np.nan
    feedback_count = int(pd.to_numeric(split_summary_row["feedback_reflections"]))
    assert segment_count == int(pd.to_numeric(split_summary_row["unique_modeling_segments"]))
    experimental_split_rows.append({
        "split": split_name,
        "documents": document_count,
        "normalized authors": normalized_author_count,
        "unique segments": segment_count,
        "share of split segments (%)": round(100 * segment_count / local_corpus_stats["segments"], 2),
        "candidate rows": candidate_count,
        "feedback-available documents": feedback_count,
    })

experimental_split_summary = pd.DataFrame(experimental_split_rows)
experimental_split_totals = {
    "documents": int(experimental_split_summary["documents"].sum()),
    "normalized authors": int(local_segments["normalized_author"].nunique()) if sibling_sources_available else np.nan,
    "unique segments": int(experimental_split_summary["unique segments"].sum()),
    "candidate rows": int(experimental_split_summary["candidate rows"].sum()),
    "feedback-available documents": int(experimental_split_summary["feedback-available documents"].sum()),
}
experimental_split_summary.loc[len(experimental_split_summary)] = {
    "split": "Total",
    "documents": experimental_split_totals["documents"],
    "normalized authors": experimental_split_totals["normalized authors"],
    "unique segments": experimental_split_totals["unique segments"],
    "share of split segments (%)": 100.0,
    "candidate rows": experimental_split_totals["candidate rows"],
    "feedback-available documents": experimental_split_totals["feedback-available documents"],
}

assert experimental_split_totals["documents"] == local_corpus_stats["documents"] == 239
assert experimental_split_totals["normalized authors"] == corpus_stats["authors"] == 63
assert experimental_split_totals["unique segments"] == local_corpus_stats["segments"] == 9779
assert experimental_split_totals["candidate rows"] == len(local_candidates) == 9835
assert experimental_split_totals["feedback-available documents"] == 22
assert experimental_split_summary.loc[experimental_split_summary["split"].eq("test"), "feedback-available documents"].item() == 22

display(experimental_split_summary)

if sibling_sources_available:
    historical_segment_ids = set(full_single["segment_id"])
    experimental_segment_ids = set(local_segments["segment_id"])
    overlap_summary = pd.DataFrame([
        {"comparison": "Present in both historical descriptive and current experimental views", "unique segments": len(historical_segment_ids & experimental_segment_ids)},
        {"comparison": "Historical descriptive only", "unique segments": len(historical_segment_ids - experimental_segment_ids)},
        {"comparison": "Current experimental view only", "unique segments": len(experimental_segment_ids - historical_segment_ids)},
    ])
    assert len(historical_segment_ids & experimental_segment_ids) == 9723
    assert len(historical_segment_ids - experimental_segment_ids) == 1902
    assert len(experimental_segment_ids - historical_segment_ids) == 56
    display(overlap_summary)
    print("The 9,723 figure is the overlap between source views; the current experimental split contains 9,779 unique segments.")
    print(f"Normalized author names appearing in more than one existing split: {normalized_authors_cross_split}.")
    if normalized_authors_cross_split:
        print("The current assignment grouped source author IDs; normalized-name aliases were not used to enforce partition separation.")

print("Counts above describe assigned eligible rows, not raw corpus coverage.")

,split,documents,normalized authors,unique segments,share of split segments (%),candidate rows,feedback-available documents
0,train,181,51,7065,72.25,7107,0
1,dev,34,10,1811,18.52,1823,0
2,test,24,6,903,9.23,905,22
3,Total,239,63,9779,100.00,9835,22


,comparison,unique segments
0,Present in both historical descriptive and cur...,9723
1,Historical descriptive only,1902
2,Current experimental view only,56


The 9,723 figure is the overlap between source views; the current experimental split contains 9,779 unique segments.
Normalized author names appearing in more than one existing split: 4.
The current assignment grouped source author IDs; normalized-name aliases were not used to enforce partition separation.
Counts above describe assigned eligible rows, not raw corpus coverage.


## 12. Timing of Single- and Multi-Creator Segments

Use the creation timestamps for annotation records referenced by the current experimental candidate rows. For each segment, the latest referenced `created_at` is used as a **completion-time proxy**; it is not a formal adjudication or approval timestamp. Compare single- versus multi-creator timing overall and within each split, including the share of all multi/single segment pairs where the multi-creator segment has the earlier timestamp. Only aggregate dates and counts are displayed.

In [54]:
if sibling_sources_available and annotation_ready:
    split_annotation_refs = pd.concat(
        [
            frame[["segment_id", "annotation_ids"]].assign(split=split_name)
            for split_name, frame in split_wide.items()
        ],
        ignore_index=True,
    )
    split_annotation_refs = split_annotation_refs.drop_duplicates()
    split_annotation_refs["annotation_id"] = (
        split_annotation_refs["annotation_ids"].fillna("").str.split("|")
    )
    split_annotation_refs = (
        split_annotation_refs.explode("annotation_id")
        .drop(columns="annotation_ids")
    )
    split_annotation_refs["annotation_id"] = (
        split_annotation_refs["annotation_id"].astype("string").str.strip()
    )
    split_annotation_refs = split_annotation_refs.loc[
        split_annotation_refs["annotation_id"].ne("")
    ].drop_duplicates(["split", "segment_id", "annotation_id"])

    annotation_time_lookup = canonical_annotations[
        ["annotation_id", "creator", "created_at"]
    ].copy()
    annotation_time_lookup["created_at"] = pd.to_datetime(
        annotation_time_lookup["created_at"], dayfirst=True, errors="coerce"
    )
    split_annotation_events = split_annotation_refs.merge(
        annotation_time_lookup,
        on="annotation_id",
        how="left",
        validate="many_to_one",
    )
    assert split_annotation_events["creator"].notna().all(), "Some candidate annotation references have no creator"
    assert split_annotation_events["created_at"].notna().all(), "Some referenced annotations have no creation timestamp"

    segment_completion = (
        split_annotation_events.groupby(["split", "segment_id"], as_index=False)
        .agg(
            distinct_creators=("creator", "nunique"),
            completion_proxy=("created_at", "max"),
        )
    )
    segment_completion["coding_group"] = np.where(
        segment_completion["distinct_creators"].eq(1), "single creator", "multiple creators"
    )
    assert segment_completion["segment_id"].nunique() == local_corpus_stats["segments"]
    assert segment_completion["distinct_creators"].gt(0).all()

    def summarize_completion_timing(frame: pd.DataFrame, split_label: str) -> dict[str, object]:
        single_times = frame.loc[
            frame["coding_group"].eq("single creator"), "completion_proxy"
        ]
        multi_times = frame.loc[
            frame["coding_group"].eq("multiple creators"), "completion_proxy"
        ]
        single_median = single_times.median()
        multi_median = multi_times.median()
        multi_values = multi_times.to_numpy(dtype="datetime64[ns]")
        single_values = single_times.to_numpy(dtype="datetime64[ns]")
        pairwise_earlier_pct = (
            100 * np.less(multi_values[:, None], single_values[None, :]).mean()
            if len(multi_values) and len(single_values)
            else np.nan
        )
        before_single_median_pct = (
            100 * multi_times.lt(single_median).mean()
            if len(multi_times) and pd.notna(single_median)
            else np.nan
        )
        return {
            "split": split_label,
            "single-creator segments": len(single_times),
            "multi-creator segments": len(multi_times),
            "single median completion proxy": str(single_median)[:10] if pd.notna(single_median) else "Unavailable",
            "multi median completion proxy": str(multi_median)[:10] if pd.notna(multi_median) else "Unavailable",
            "multi earlier than single median (%)": round(before_single_median_pct, 1) if pd.notna(before_single_median_pct) else np.nan,
            "multi earlier in all multi/single pairs (%)": round(pairwise_earlier_pct, 1) if pd.notna(pairwise_earlier_pct) else np.nan,
        }

    completion_timing_rows = [summarize_completion_timing(segment_completion, "All splits")]
    completion_timing_rows.extend(
        summarize_completion_timing(
            segment_completion.loc[segment_completion["split"].eq(split_name)], split_name
        )
        for split_name in ("train", "dev", "test")
    )
    completion_timing_summary = pd.DataFrame(completion_timing_rows)
    assert completion_timing_summary.loc[0, "single-creator segments"] == 8837
    assert completion_timing_summary.loc[0, "multi-creator segments"] == 942
    display(completion_timing_summary)

    completion_timing_summary.to_csv(
        OUTPUT_DIR / "experimental_split_annotation_completion_timing.csv", index=False
    )
    overall_timing = completion_timing_summary.iloc[0]
    if overall_timing["multi median completion proxy"] > overall_timing["single median completion proxy"]:
        conclusion = "Multi-creator segments were typically completed later by this timestamp proxy."
    else:
        conclusion = "Multi-creator segments were typically completed earlier by this timestamp proxy."
    print(conclusion)
    print(
        "The pairwise percentage compares every multi-creator segment with every single-creator segment; "
        "values below 50% mean multi-creator segments were not mostly earlier."
    )
else:
    print("Completion-timing reconstruction is unavailable without the sibling annotation source.")

,split,single-creator segments,multi-creator segments,single median completion proxy,multi median completion proxy,multi earlier than single median (%),multi earlier in all multi/single pairs (%)
0,All splits,8837,942,2023-12-16,2023-12-29,39.8,32.6
1,train,6292,773,2023-12-18,2023-12-29,41.0,34.2
2,dev,1683,128,2023-12-01,2023-12-17,12.5,33.1
3,test,862,41,2023-12-01,2024-01-14,26.8,19.0


Multi-creator segments were typically completed later by this timestamp proxy.
The pairwise percentage compares every multi-creator segment with every single-creator segment; values below 50% mean multi-creator segments were not mostly earlier.


## 13. Experimental Split and Annotation Tables

These two tables summarize the current experimental view only. The split table reports normalized author names; per-split counts overlap because four normalized authors occur in multiple partitions, so only the overall total is unique. The annotation table counts distinct creators among the annotations referenced by candidate rows. Creator/editor identities and creation-time proxies are labeled separately to avoid treating edits as independent coding or timestamps as adjudication dates.

In [56]:
if sibling_sources_available and annotation_ready:
    experimental_split_table = experimental_split_summary[
        [
            "split",
            "documents",
            "normalized authors",
            "unique segments",
            "candidate rows",
            "feedback-available documents",
        ]
    ].copy()
    experimental_split_table = experimental_split_table.rename(columns={
        "split": "Split",
        "documents": "Documents",
        "normalized authors": "Normalized authors",
        "unique segments": "Unique segments",
        "candidate rows": "Candidate rows",
        "feedback-available documents": "Feedback-available documents",
    })

    split_denominator = int(local_corpus_stats["segments"])
    creator_count_by_segment = segment_completion.groupby("segment_id")["distinct_creators"].first()
    creator_multiplicity_counts = creator_count_by_segment.value_counts().sort_index()
    single_creator_n = int(creator_count_by_segment.eq(1).sum())
    two_creator_n = int(creator_count_by_segment.eq(2).sum())
    three_plus_creator_n = int(creator_count_by_segment.ge(3).sum())
    multi_creator_n = int(creator_count_by_segment.gt(1).sum())

    def format_count_share(count: int, denominator: int) -> str:
        percentage = 100 * count / denominator
        precision = 2 if percentage < 1 else 1
        return f"{count:,} ({percentage:.{precision}f}%)"

    overall_timing_row = completion_timing_summary.loc[
        completion_timing_summary["split"].eq("All splits")
    ].iloc[0]
    creator_count_total = int(canonical_annotations["creator"].nunique(dropna=True))
    creator_editor_identity_total = len(coder_values)

    annotation_table = pd.DataFrame([
        {"Aspect": "Annotator pool", "Statistic": "Distinct annotation creators", "Value": str(creator_count_total)},
        {"Aspect": "Annotator pool", "Statistic": "Distinct creator/editor identities (includes editor-only identity)", "Value": str(creator_editor_identity_total)},
        {"Aspect": "Coverage (N = 9,779 segments)", "Statistic": "Coded by 1 creator", "Value": format_count_share(single_creator_n, split_denominator)},
        {"Aspect": "Coverage (N = 9,779 segments)", "Statistic": "Coded by 2 creators", "Value": format_count_share(two_creator_n, split_denominator)},
        {"Aspect": "Coverage (N = 9,779 segments)", "Statistic": "Coded by 3+ creators", "Value": format_count_share(three_plus_creator_n, split_denominator)},
        {"Aspect": "Coverage (N = 9,779 segments)", "Statistic": "Coded by >1 creator", "Value": format_count_share(multi_creator_n, split_denominator)},
        {"Aspect": "Timing proxy", "Statistic": "Median latest annotation creation date: 1 creator", "Value": str(overall_timing_row["single median completion proxy"])},
        {"Aspect": "Timing proxy", "Statistic": "Median latest annotation creation date: >1 creator", "Value": str(overall_timing_row["multi median completion proxy"])},
        {"Aspect": "Timing proxy", "Statistic": ">1-creator segment earlier in all multi/single pairs", "Value": f"{overall_timing_row['multi earlier in all multi/single pairs (%)']:.1f}%"},
    ])

    assert creator_count_total == 5
    assert creator_editor_identity_total == 6
    assert single_creator_n == 8837
    assert two_creator_n + three_plus_creator_n == multi_creator_n == 942
    assert creator_count_by_segment.size == split_denominator

    print("Table A. Experimental split composition")
    display(experimental_split_table)
    print("Table B. Annotation coverage and timing")
    display(annotation_table)
    print("Interpretation: most current-view segments have one creator; multi-creator segments are not mostly earlier by creation-time proxy.")

    experimental_split_table.to_csv(
        OUTPUT_DIR / "experimental_split_descriptive_table.csv", index=False
    )
    annotation_table.to_csv(
        OUTPUT_DIR / "experimental_split_annotation_descriptive_table.csv", index=False
    )

    split_tex = experimental_split_table.to_latex(index=False, escape=True)
    annotation_tex = annotation_table.to_latex(index=False, escape=True)
    (OUTPUT_DIR / "experimental_split_descriptive_table.tex").write_text(
        "% Experimental split composition; normalized author counts overlap across splits.\n" + split_tex,
        encoding="utf-8",
    )
    (OUTPUT_DIR / "experimental_split_annotation_descriptive_table.tex").write_text(
        "% Creator counts use referenced annotations; latest created_at is a timing proxy, not adjudication.\n" + annotation_tex,
        encoding="utf-8",
    )
    print(f"CSV and LaTeX tables saved under: {OUTPUT_DIR}")
else:
    print("The experimental split and annotation tables require the sibling annotation sources.")

Table A. Experimental split composition


,Split,Documents,Normalized authors,Unique segments,Candidate rows,Feedback-available documents
0,train,181,51,7065,7107,0
1,dev,34,10,1811,1823,0
2,test,24,6,903,905,22
3,Total,239,63,9779,9835,22


Table B. Annotation coverage and timing


,Aspect,Statistic,Value
0,Annotator pool,Distinct annotation creators,5
1,Annotator pool,Distinct creator/editor identities (includes e...,6
2,"Coverage (N = 9,779 segments)",Coded by 1 creator,"8,837 (90.4%)"
3,"Coverage (N = 9,779 segments)",Coded by 2 creators,938 (9.6%)
4,"Coverage (N = 9,779 segments)",Coded by 3+ creators,4 (0.04%)
5,"Coverage (N = 9,779 segments)",Coded by >1 creator,942 (9.6%)
6,Timing proxy,Median latest annotation creation date: 1 creator,2023-12-16
7,Timing proxy,Median latest annotation creation date: >1 cre...,2023-12-29
8,Timing proxy,>1-creator segment earlier in all multi/single...,32.6%


Interpretation: most current-view segments have one creator; multi-creator segments are not mostly earlier by creation-time proxy.
CSV and LaTeX tables saved under: /Users/quynguyen/Documents/CodingProjects/prebi_v1/artifacts/corpus_annotation_descriptives
